<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week8_Lecture_Wrangling_Reshaping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

### Week 7-8: Data Wrangling
---

**Question:**
- What capabilities does Python provide to wrangle and tranform data? (continue) 

**Objectives:**
- Discretize data
- Random sample
- Compute indicator variables
- Reshape and pivot data

In [ ]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 20
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

## Exercise 1: 
### For the titanic data, create a new column representing the age category of passengers such as child, youth, middle-aged, elderly, etc. Before binning, fill up missing age values.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include='O')

In [ ]:
df.describe()

In [ ]:
df.Age.median()

In [ ]:
df.Age.mean()

In [ ]:
df.Age.min()

In [ ]:
df.Age.max()

In [ ]:
df.Age.isnull().sum()

In [ ]:
bins = [0, 12, 35, 60, 100]

In [ ]:
cats = pd.cut(df.Age, bins, labels = ['child', 'youth', 'middle', 'elderly'])

In [ ]:
cats

In [ ]:
df['Age-cats'] = cats

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df['Age-cats'].isnull().sum()

In [ ]:
pd.value_counts(df['Age-cats']).plot.bar()

In [ ]:
pd.value_counts(df['Age-cats']).sum()

In [ ]:
age_mean_fill = df.Age.fillna(df.Age.mean())

In [ ]:
age_mean_fill

In [ ]:
age_mean_fill.isnull().sum()

In [ ]:
df['Age_mean_fill_cats'] = pd.cut(age_mean_fill, bins=[0, 12, 35, 60, 100], labels=['child', 'youth', 'middle', 'elderly'])

In [ ]:
df.head()

In [ ]:
df.Age_mean_fill_cats.isnull().sum()

In [ ]:
df.Age.mean()

In [ ]:
177/891

In [ ]:
pd.value_counts(df.Age_mean_fill_cats)

In [ ]:
df.Pclass.unique()

In [ ]:
import seaborn as sns
g = sns.FacetGrid(df, row = 'Sex', col = 'Pclass')
g.map(plt.hist, 'Age')

In [ ]:
means = pd.DataFrame(np.zeros([2, 3]), index=['male', 'female'], columns=[1,2,3])

In [ ]:
means

In [ ]:
means.loc['female', 3]

In [ ]:
for s in ['male', 'female']:
    for p in [1, 2, 3]:
        means.loc[s, p] = df[(df.Sex == s) & (df.Pclass == p)].Age.mean()

In [ ]:
means

In [ ]:
df[((df.Sex == 'female') & (df.Pclass == 2) & df.Age.isnull())].Age

In [ ]:
for s in ['male', 'female']:
    for p in [1, 2, 3]:
        df.loc[(df.Sex == s) & (df.Pclass == p) & (df.Age.isnull()), 'Age'] = means.loc[s, p]

In [ ]:
df[((df.Sex == 'female') & (df.Pclass == 2) & df.Age.isnull())].Age

In [ ]:
df.describe()

In [ ]:
df['Age_diff_means_cats'] = pd.cut(df.Age, bins=[0, 12, 35, 60, 100], labels = ['child', 'youth', 'middle', 'elderly'])

In [ ]:
pd.value_counts(df.Age_diff_means_cats).plot.bar()

In [ ]:
pd.value_counts(df.Age_diff_means_cats)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Computing Indicator/Dummy Variables
Another type of transformation for statistical modeling or machine learning applications
is converting a categorical variable into a “dummy” or “indicator” matrix. If a
column in a DataFrame has k distinct values, you would derive a matrix or DataFrame with k columns containing all 1s and 0s. pandas has a get_dummies function
for doing this, though devising one yourself is not difficult. Let’s return to an earlier
example DataFrame:

In [ ]:
df = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                   'data1': range(6)})
df

In [ ]:
pd.get_dummies(df['key'])

### In some cases, you may want to add a prefix to the columns in the indicator DataFrame, which can then be merged with the other data. get_dummies has a prefix argument for doing this:

In [ ]:
dummies = pd.get_dummies(df['key'], prefix='dummy')
dummies

In [ ]:
df[['data1']].join(dummies)

In [ ]:
df_with_dummy = df[['data1']].join(dummies)
df_with_dummy

In [ ]:
pd.merge(df[['data1']], dummies, left_index=True, right_index=True)

### If a row in a DataFrame belongs to multiple categories, things are a bit more complicated.

In [ ]:
mnames = ['movie_id', 'title', 'genres']

In [ ]:
from google.colab import files
files.upload()

In [ ]:

movies = pd.read_table('movies.dat', sep='::', header=None, names=mnames)

In [ ]:
movies.head()

### Adding indicator variables for each genre requires a little bit of wrangling. First, we extract the list of  unique genres in the dataset:

In [ ]:
all_genres = []
for x in movies.genres:
    all_genres.extend(x.split('|'))
genres = pd.unique(all_genres)

In [ ]:
genres

In [ ]:
len(genres)

In [ ]:
pd.get_dummies(movies['genres'])

### One way to construct the indicator DataFrame is to start with a DataFrame of all zeros:

In [ ]:
movies.shape

In [ ]:
zero_matrix = np.zeros((movies.shape[0], len(genres)))
dummies = pd.DataFrame(zero_matrix, columns=genres)
dummies

### Now, iterate through each movie and set entries in each row of dummies to 1. To do this, we use the dummies.columns to compute the column indices for each genre:

In [ ]:
gen = movies.genres[0]
gen

In [ ]:
gen.split('|')

In [ ]:
dummies.columns.get_indexer(['Animation', "Children's", 'Fantasy'])

In [ ]:
dummies.columns.get_indexer(gen.split('|'))

### Then, we can use .iloc to set values based on these indices:

In [ ]:
for i, gen in enumerate(movies.genres):
    cats = gen.split("|")
    indices = dummies.columns.get_indexer(cats)
    dummies.iloc[i, indices] = 1

In [ ]:
dummies.head()

### Then, as before, you can combine this with movies:

In [ ]:
dummies.add_prefix('dummy_')

In [ ]:
movies.join(dummies.add_prefix('dummy_'))

In [ ]:
movies_windic = movies.join(dummies.add_prefix('Genre_'))

In [ ]:
movies_windic.head()

In [ ]:
movies_windic.iloc[0]

### A useful recipe for statistical applications is to combine get_dummies with a discretization function like cut:

In [ ]:
np.random.seed(12345)
values = np.random.rand(10)
values

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

In [ ]:
labels =['first', 'second', 'third', 'fourth', 'fifth']

In [ ]:
pd.cut(values, bins, labels=labels)

In [ ]:
pd.get_dummies(pd.cut(values, bins, labels=labels))

# Lecture on Friday

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## String Manipulation
Python has long been a popular raw data manipulation language in part due to its
ease of use for string and text processing. Most text operations are made simple with
the string object’s built-in methods. For more complex pattern matching and text
manipulations, regular expressions may be needed. pandas adds to the mix by enabling
you to apply string and regular expressions concisely on whole arrays of data,
additionally handling the annoyance of missing data.

### String Object Methods

In many string munging and scripting applications, built-in string methods are sufficient.
As an example, a comma-separated string can be broken into pieces with
split:

In [ ]:
val = 'a,b,  guido'
[s.strip() for s in val.split(',')]

### split is often combined with strip to trim whitespace (including line breaks):

In [ ]:
pieces = [x.strip() for x in val.split(',')]
pieces

These substrings could be concatenated together with a two-colon delimiter using
addition:

In [ ]:
pieces=  val.split(',')
pieces

In [ ]:
first, second, third = pieces

first + ',' + second + ',' + third

### But this isn’t a practical generic method. A faster and more Pythonic way is to pass a list or tuple to the join method on the string '::':

In [ ]:
','.join(pieces)

In [ ]:
sent = "Other methods are concerned with locating substrings"
sent

In [ ]:
wds = sent.split(" ")
wds

In [ ]:
" ".join(wds)

### Other methods are concerned with locating substrings. Using Python’s in keyword is the best way to detect a substring, though index and find can also be used:

In [ ]:
val

In [ ]:
'guido' in val

In [ ]:
'a,b, ' in val

In [ ]:
val.index('  guido')

In [ ]:
val.index('b,')

In [ ]:
val.find('b,')

### Note the difference between find and index is that index raises an exception if the string isn’t found (versus returning –1):

In [ ]:
val.index(':')

### Relatedly, count returns the number of occurrences of a particular substring:

In [ ]:
val = "Relatedly, count returns the number of occurrences of a particular substring"

In [ ]:
val.count('ing')

### replace will substitute occurrences of one pattern for another. It is commonly used to delete patterns, too, by passing an empty string:

In [ ]:
val[0] = 'r'

In [ ]:
newval = val.replace('R', 'r')

In [ ]:
val

In [ ]:
newval

In [ ]:
val.replace(',', '')

### Regular Expressions
Regular expressions provide a flexible way to search or match (often more complex)
string patterns in text. A single expression, commonly called a regex, is a string
formed according to the regular expression language. Python’s built-in re module is
responsible for applying regular expressions to strings; I’ll give a number of examples
of its use here.

The re module functions fall into three categories: pattern matching, substitution,
and splitting. Naturally these are all related; a regex describes a pattern to locate in the
text, which can then be used for many purposes.

Suppose we wanted to split a string with a variable number of whitespace characters
(tabs, spaces, and newlines). The regex describing one or more whitespace characters
is \s+:

In [ ]:
import re

text = "foo    bar\t baz  \tqux"
text

In [ ]:
text.split(" ")

In [ ]:
re.split("\s+", text)

In [ ]:
re.split('\s+', text)

In [ ]:
text.split()

### When you call re.split('\s+', text), the regular expression is first compiled, and then its split method is called on the passed text. You can compile the regex yourself with re.compile, forming a reusable regex object:

In [ ]:
regex = re.compile('\s+')
regex.split(text)

### If, instead, you wanted to get a list of all patterns matching the regex, you can use the findall method:

In [ ]:
text

In [ ]:
regex.findall(text)

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

### match and search are closely related to findall. While findall returns all matches in a string, search returns only the first match. More rigidly, match only matches at the beginning of the string. As a less trivial example, let’s consider a block of text and a regular expression capable of identifying most email addresses:

In [ ]:
text = """Dave dave@google.com
Steve steve@gmail.company
Rob rob@gmail.com
Ryan ryan@yahoo.com
"""

text

In [ ]:
pattern = r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}\b'

In [ ]:
import re
# re.IGNORECASE makes the regex case-insensitive
regex = re.compile(pattern, flags=re.IGNORECASE)

### Using findall on the text produces a list of the email addresses:

In [ ]:
regex.findall(text)

### search returns a special match object for the first email address in the text. For the preceding regex, the match object can only tell us the start and end position of the pattern in the string:

In [ ]:
m = regex.search(text)
m

In [ ]:
text[m.start():m.end()]

In [ ]:
regex.match(text)

### regex.match returns None, as it only will match if the pattern occurs at the start of the string:

In [ ]:
print(regex.match(text))

### Relatedly, sub will return a new string with occurrences of the pattern replaced by the a new string:

In [ ]:
print(regex.sub('REDACTED', text))

### Suppose you wanted to find email addresses and simultaneously segment each address into its three components: username, domain name, and domain suffix. To do this, put parentheses around the parts of the pattern to segment:

In [ ]:
pattern = r'([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})'
regex = re.compile(pattern, flags=re.IGNORECASE)

### A match object produced by this modified regex returns a tuple of the pattern components with its groups method:

In [ ]:
m = regex.match('wesm@bright.net')
m.groups()

### findall returns a list of tuples when the pattern has groups:

In [ ]:
regex.findall(text)

### sub also has access to groups in each match using special symbols like \1 and \2. The symbol \1 corresponds to the first matched group, \2 corresponds to the second, and so forth:

In [ ]:
print(regex.sub(r'Username: \1, Domain: \2, Suffix: \3', text))

### Vectorized String Functions in pandas
Cleaning up a messy dataset for analysis often requires a lot of string munging and
regularization. To complicate matters, a column containing strings will sometimes
have missing data:

In [ ]:
data = {'Dave': 'dave@google.com', 'Steve': 'steve@gmail.com',
        'Rob': 'rob@gmail.com', 'Wes':np.nan}

In [ ]:
data = pd.Series(data)
data

In [ ]:
data.isnull()

### You can apply string and regular expression methods (passing a lambda or other function) to each value using data.map, but it will fail on the NA (null) values. To cope with this, Series has array-oriented methods for string operations that skip NA values. These are accessed through Series’s str attribute; for example, we could check whether each email address has 'gmail' in it with str.contains:

In [ ]:
data.map(lambda x: 'gmail' in x)

In [ ]:
data.str.contains('gmail')

### Regular expressions can be used, too, along with any re options like IGNORECASE:

In [ ]:
pattern

In [ ]:
data.str.findall(pattern, flags=re.IGNORECASE)

### There are a couple of ways to do vectorized element retrieval. Either use str.get or index into the str attribute:

In [ ]:
matches = data.str.match(pattern, flags=re.IGNORECASE)
matches

In [ ]:
data

In [ ]:
data.str.get(1)

In [ ]:
data.str[0]

### You can similarly slice strings using this syntax:

In [ ]:
data.str[:5]

# Exercise: Load the titanic data and perform the following data transformations (If time permits)
1. Extract titles from names.
2. Consolidate titles to a small list of common titles.
3. Add a new column indiating the titles of passergers.
4. Convert the titles columns to numeric values.

In [ ]:
df = pd.read_csv("datasets/titanic/train.csv")

In [ ]:
df.head()

In [ ]:
import re
pattern = r' ([A-Za-z]+)\.'

In [ ]:
regx = re.compile(pattern, flags=re.IGNORECASE)

In [ ]:
df.iloc[2].Name

In [ ]:
regx.findall(df.iloc[2].Name)

In [ ]:
df.Name.str.findall(pattern)

In [ ]:
df.Name.str.extract(pattern, expand=False)

In [ ]:
df['Title'] = df.Name.str.extract(pattern, expand=False)

In [ ]:
df['Title'].value_counts()

In [ ]:
pd.crosstab(df['Title'], df['Sex'])

We can replace many titles with a more common name or classify them as `Rare`.

In [ ]:
title_repl = {'Col':'Mr', 'Major':'Master', 'Mlle':'Miss', 'Mme':'Mrs', 'Sir':'Mr', 'Capt':'Master', 'Countess':'Mrs', 
'Jonkheer':'Mr', 'Don':'Master', 'Ms':'Miss', 'Lady':'Miss'}

In [ ]:
df['Title_repl'] = df['Title'].replace(title_repl)

In [ ]:
df['Title_repl'].unique()

In [ ]:
df['Title_repl'].value_counts()

In [ ]:
df[['Title_repl', 'Survived']].groupby(['Title_repl'], as_index=False).mean()

We can convert the categorical titles to ordinal.

In [ ]:
title_mapping = {'Dr':0, 'Master':1, 'Miss':2, 'Mr':3, 'Mrs':4, 'Rev':5}

In [ ]:
df['Title_num'] = df['Title_repl'].map(title_mapping)

In [ ]:
df['Title_num']

In [ ]:
df['Title_num'].isnull().sum()

In [ ]:
df.head(3)

Now we can safely drop the Name feature from the dataset. We also do not need the PassengerId feature in the dataset.

In [ ]:
df = df.drop(['Name', 'PassengerId'], axis=1)

In [ ]:
df.head(3)